# Outline

- Perform Optuna with Model selection
- Then choose a good model and then do hyper parameter tunning with it
- Make a pickle file for pipeline (used in the Feature Selection) and for Model

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
X_train = pd.read_csv('../Feature Selection/X_train_houses.csv',index_col=0,)
X_test = pd.read_csv('../Feature Selection/X_test_houses.csv',index_col=0)
y_train = pd.read_csv('../Feature Selection/y_train_houses.csv',index_col=0)
y_test = pd.read_csv('../Feature Selection/y_test_houses.csv',index_col=0)

In [3]:
X_train.head()

,Main Location,Area(Marla),Bath(s),Bedroom(s),Servant Quarters,Kitchens,Store Rooms,Storey Unit,IsPrimeLoc,SolarInstalled,WaterBore,CornerHouse,Property era
906,11.633882,2.079442,5.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,5
3379,4.004817,2.397895,6.0,5.0,1.0,2.0,1.0,2.0,False,False,False,False,5
2094,3.778618,1.791759,4.0,3.0,1.0,2.0,1.0,2.0,True,False,False,True,5
218,9.283625,2.708050,6.0,6.0,1.0,1.0,1.0,2.0,True,False,False,False,5
3455,7.172108,1.609438,5.0,5.0,1.0,2.0,1.0,2.0,True,False,False,False,5


In [4]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 2878 entries, 906 to 1246
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Main Location     2878 non-null   float64
 1   Area(Marla)       2878 non-null   float64
 2   Bath(s)           2878 non-null   float64
 3   Bedroom(s)        2878 non-null   float64
 4   Servant Quarters  2878 non-null   float64
 5   Kitchens          2878 non-null   float64
 6   Store Rooms       2878 non-null   float64
 7   Storey Unit       2878 non-null   float64
 8   IsPrimeLoc        2878 non-null   bool   
 9   SolarInstalled    2878 non-null   bool   
 10  WaterBore         2878 non-null   bool   
 11  CornerHouse       2878 non-null   bool   
 12  Property era      2878 non-null   int64  
dtypes: bool(4), float64(8), int64(1)
memory usage: 236.1 KB


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

In [6]:
y_train_log = np.log1p(y_train)

With y_train log transformed

In [7]:
clf = RandomForestRegressor(random_state=42)
clf.fit(X_train,y_train_log)
y_pred = clf.predict(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [8]:
y_pred = np.expm1(y_pred)
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9352733235118627
Mean absolute error 1.3011855942634751


Without y_train log transformed

In [9]:
clf = RandomForestRegressor(random_state=42)
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [10]:
print('r2 Score',r2_score(y_test,y_pred))
print('Mean absolute error',mean_absolute_error(y_test,y_pred))

r2 Score 0.9282135045471744
Mean absolute error 1.3246984020383226


With y_train as log_transformed, it is giving good results

## Optuna (for Model Selection)

In [11]:
from sklearn.model_selection import StratifiedKFold

In [12]:
import optuna
from optuna.visualization import plot_contour,plot_optimization_history,plot_parallel_coordinate,plot_param_importances

In [13]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

def Multiple_Ml_Objective(trial):

    # 1. Select the Model
    regressor_name = trial.suggest_categorical('regressor', ['Random Forest', 'Extra Trees','Xgboost'])

    # 2. Random Forest Branch
    if regressor_name == 'Random Forest':
        n_estimators = trial.suggest_int('rf_n_estimators', 100, 800, step=100) 
        criterion = trial.suggest_categorical('rf_criterion', ['squared_error', 'absolute_error'])
        
        max_depth_option = trial.suggest_categorical('rf_max_depth_option', ['auto', 'fixed'])
        max_depth = None if max_depth_option == 'auto' else trial.suggest_int('rf_max_depth', 10, 150, step=10)
        
        min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 30)
        min_samples_leaf = trial.suggest_int('rf_min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('rf_max_features', ['sqrt', 'log2', 1.0, 0.3, 0.5, 0.75])
        max_samples = trial.suggest_float('rf_max_samples', 0.4, 0.95)
        
        # NEW: Tree Pruning & Size Control
        max_leaf_nodes = trial.suggest_int('rf_max_leaf_nodes', 50, 2000, log=True)
        ccp_alpha = trial.suggest_float('rf_ccp_alpha', 0.0, 0.1)

        model = RandomForestRegressor(
            n_estimators=n_estimators, 
            criterion=criterion, 
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features, 
            max_samples=max_samples,
            max_leaf_nodes=max_leaf_nodes,
            ccp_alpha=ccp_alpha,
            n_jobs=-1,
            random_state=42
        )

    # 3. Extra Trees Branch
    elif regressor_name == 'Extra Trees':
        n_estimators = trial.suggest_int('et_n_estimators', 100, 800, step=100)
        criterion = trial.suggest_categorical('et_criterion', ['squared_error', 'absolute_error'])
        
        max_depth_option = trial.suggest_categorical('et_max_depth_option', ['auto', 'fixed'])
        max_depth = None if max_depth_option == 'auto' else trial.suggest_int('et_max_depth', 10, 150, step=10)
        
        min_samples_split = trial.suggest_int('et_min_samples_split', 2, 30)
        min_samples_leaf = trial.suggest_int('et_min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('et_max_features', ['sqrt', 'log2', 1.0, 0.3, 0.5, 0.75])
        
        # NEW: Bootstrapping & Pruning 
        bootstrap = trial.suggest_categorical('et_bootstrap', [True, False])
        max_samples = trial.suggest_float('et_max_samples', 0.4, 0.95) if bootstrap else None
        max_leaf_nodes = trial.suggest_int('et_max_leaf_nodes', 50, 2000, log=True)
        ccp_alpha = trial.suggest_float('et_ccp_alpha', 0.0, 0.1)

        model = ExtraTreesRegressor(
            n_estimators=n_estimators, 
            criterion=criterion, 
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features, 
            bootstrap=bootstrap,
            max_samples=max_samples,
            max_leaf_nodes=max_leaf_nodes,
            ccp_alpha=ccp_alpha,
            n_jobs=-1,
            random_state=42
        )

    # 4. XGBoost Branch
    elif regressor_name == 'Xgboost':
        n_estimators = trial.suggest_int('xgb_n_estimators', 200, 1000, step=100)
        max_depth = trial.suggest_int('xgb_max_depth', 3, 20)
        learning_rate = trial.suggest_float('xgb_learning_rate', 1e-4, 0.3, log=True)
        subsample = trial.suggest_float('xgb_subsample', 0.5, 1.0)
        colsample_bytree = trial.suggest_float('xgb_colsample_bytree', 0.4, 1.0)
        gamma = trial.suggest_float('xgb_gamma', 0.0, 10.0)
        
        # NEW: L1/L2 Regularization and Node Splitting Control
        min_child_weight = trial.suggest_int('xgb_min_child_weight', 1, 20)
        reg_alpha = trial.suggest_float('xgb_reg_alpha', 1e-5, 10.0, log=True) # L1
        reg_lambda = trial.suggest_float('xgb_reg_lambda', 1e-5, 10.0, log=True) # L2
        colsample_bylevel = trial.suggest_float('xgb_colsample_bylevel', 0.4, 1.0)

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            colsample_bylevel=colsample_bylevel,
            gamma=gamma,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            n_jobs=-1,
            random_state=42
        )

    # 5. Cross Validation
    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_results = cross_validate(
        model,
        X_train,
        y_train['Price(Cr)'], # Note: Ensure you use y_train_log here if you want to keep optimizing in log-space!
        scoring='neg_mean_absolute_error',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1 # Speeds up cross-validation
    )

    # 6. Extract Metrics
    train_score_mean = cv_results['train_score'].mean()
    val_score_mean = cv_results['test_score'].mean()

    # 7. Log Custom Attributes to Optuna Dashboard
    trial.set_user_attr('train_score_mean', train_score_mean)
    trial.set_user_attr('train_score_std', cv_results['train_score'].std())
    trial.set_user_attr('test_score_std', cv_results['test_score'].std())
    trial.set_user_attr('overfitting_gap', train_score_mean - val_score_mean)

    return (-val_score_mean)

In [14]:
multiple_ml_study = optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(),study_name="Model_Selection")

[I 2026-05-26 21:50:35,700] A new study created in memory with name: Model_Selection


In [16]:
multiple_ml_study.optimize(Multiple_Ml_Objective,n_trials=50)

[I 2026-05-26 21:51:21,978] Trial 10 finished with value: 1.56396131973355 and parameters: {'regressor': 'Xgboost', 'xgb_n_estimators': 1000, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.2606865728411196, 'xgb_subsample': 0.5107049401223743, 'xgb_colsample_bytree': 0.9957439645656996, 'xgb_gamma': 0.7060356378513619, 'xgb_min_child_weight': 3, 'xgb_reg_alpha': 1.8441232126107522e-05, 'xgb_reg_lambda': 1.2577249791041657e-05, 'xgb_colsample_bylevel': 0.9752729303348304}. Best is trial 10 with value: 1.56396131973355.
[I 2026-05-26 21:51:23,481] Trial 11 finished with value: 1.534285530429339 and parameters: {'regressor': 'Xgboost', 'xgb_n_estimators': 1000, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.20428850487726383, 'xgb_subsample': 0.5187225706918106, 'xgb_colsample_bytree': 0.9250293533727567, 'xgb_gamma': 0.470319678292733, 'xgb_min_child_weight': 1, 'xgb_reg_alpha': 1.1234123466695048e-05, 'xgb_reg_lambda': 1.0040981928716116e-05, 'xgb_colsample_bylevel': 0.966079471693337}. Best is

In [17]:
df = multiple_ml_study.trials_dataframe()

In [18]:
df.head()

,number,value,datetime_start,datetime_complete,duration,params_et_bootstrap,params_et_ccp_alpha,params_et_criterion,params_et_max_depth,params_et_max_depth_option,...,params_xgb_min_child_weight,params_xgb_n_estimators,params_xgb_reg_alpha,params_xgb_reg_lambda,params_xgb_subsample,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
0,0,6.870116,2026-05-26 21:50:37.576677,2026-05-26 21:50:45.890689,0 days 00:00:08.314012,NaN,NaN,NaN,NaN,NaN,...,12.0,800.0,0.000798,1.737542,0.860894,0.011207,0.243946,-6.858909,0.072926,COMPLETE
1,1,6.950157,2026-05-26 21:50:45.892013,2026-05-26 21:50:47.015670,0 days 00:00:01.123657,NaN,NaN,NaN,NaN,NaN,...,18.0,500.0,0.231127,0.012020,0.861871,0.008954,0.247279,-6.941203,0.072994,COMPLETE
2,2,3.396478,2026-05-26 21:50:47.016907,2026-05-26 21:50:49.542796,0 days 00:00:02.525889,True,0.034979,squared_error,NaN,auto,...,NaN,NaN,NaN,NaN,NaN,0.043411,0.208593,-3.353067,0.024810,COMPLETE
3,3,2.177262,2026-05-26 21:50:49.543581,2026-05-26 21:50:51.588728,0 days 00:00:02.045147,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.111702,0.072584,-2.065560,0.023990,COMPLETE
4,4,3.081049,2026-05-26 21:50:51.589511,2026-05-26 21:50:53.978845,0 days 00:00:02.389334,False,0.012195,squared_error,130.0,fixed,...,NaN,NaN,NaN,NaN,NaN,0.067731,0.172482,-3.013318,0.027340,COMPLETE


In [19]:
df.groupby('params_regressor')['value'].mean()

params_regressor
Extra Trees      2.485812
Random Forest    2.024917
Xgboost          2.078846
Name: value, dtype: float64

In [20]:
df.groupby('params_regressor')['user_attrs_overfitting_gap'].mean()

params_regressor
Extra Trees      0.066594
Random Forest    0.093567
Xgboost          0.538100
Name: user_attrs_overfitting_gap, dtype: float64

In [21]:
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[True,True]).head()

,number,value,datetime_start,datetime_complete,duration,params_et_bootstrap,params_et_ccp_alpha,params_et_criterion,params_et_max_depth,params_et_max_depth_option,...,params_xgb_min_child_weight,params_xgb_n_estimators,params_xgb_reg_alpha,params_xgb_reg_lambda,params_xgb_subsample,user_attrs_overfitting_gap,user_attrs_test_score_std,user_attrs_train_score_mean,user_attrs_train_score_std,state
13,13,1.370462,2026-05-26 21:51:25.247931,2026-05-26 21:51:28.321854,0 days 00:00:03.073923,NaN,NaN,NaN,NaN,NaN,...,1.0,1000.0,0.000011,0.000012,0.638342,1.035899,0.074930,-0.334563,0.002133,COMPLETE
44,44,1.372480,2026-05-26 21:58:25.297413,2026-05-26 21:58:28.047439,0 days 00:00:02.750026,NaN,NaN,NaN,NaN,NaN,...,2.0,700.0,2.584255,0.023985,0.738423,0.738502,0.068265,-0.633977,0.004169,COMPLETE
48,48,1.375986,2026-05-26 21:58:43.436446,2026-05-26 21:58:45.186919,0 days 00:00:01.750473,NaN,NaN,NaN,NaN,NaN,...,2.0,400.0,0.000056,0.555464,0.915614,0.754678,0.061954,-0.621307,0.007568,COMPLETE
42,42,1.383745,2026-05-26 21:58:21.256170,2026-05-26 21:58:23.486002,0 days 00:00:02.229832,NaN,NaN,NaN,NaN,NaN,...,2.0,700.0,3.248184,0.033456,0.730522,0.722361,0.071036,-0.661384,0.006687,COMPLETE
47,47,1.385206,2026-05-26 21:58:40.843793,2026-05-26 21:58:43.434827,0 days 00:00:02.591034,NaN,NaN,NaN,NaN,NaN,...,1.0,500.0,0.000066,0.981152,0.872720,0.895850,0.076602,-0.489355,0.002947,COMPLETE


In [22]:
# Extract the best overall score (lowest MAE)
print("Best Validation MAE :", multiple_ml_study.best_value)

# Extract the parameters that achieved this score
best_params = multiple_ml_study.best_params
print("\nBest Hyperparameters Found:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Extract the custom attributes we logged for that specific trial
best_trial_attrs = multiple_ml_study.best_trial.user_attrs
print("\nBest Trial Diagnostics:")
print(f"  Training Score Mean: {best_trial_attrs['train_score_mean']}")
print(f"  Overfitting Gap: {best_trial_attrs['overfitting_gap']}")

Best Validation MAE : 1.3704620337711522

Best Hyperparameters Found:
  regressor: Xgboost
  xgb_n_estimators: 1000
  xgb_max_depth: 8
  xgb_learning_rate: 0.022018625744799274
  xgb_subsample: 0.6383420890153424
  xgb_colsample_bytree: 0.9894220493068537
  xgb_gamma: 0.2722178959431944
  xgb_min_child_weight: 1
  xgb_reg_alpha: 1.141747055232675e-05
  xgb_reg_lambda: 1.2015357979076116e-05
  xgb_colsample_bylevel: 0.9949975739463981

Best Trial Diagnostics:
  Training Score Mean: -0.3345629904933947
  Overfitting Gap: 1.0358990432777575


<br>

*Xgboost is giving better results so we will choose it* 

## Objective Function for XGboost only

In [23]:
from sklearn.model_selection import KFold

In [29]:
def XGBoost_Anti_Overfit_Objective(trial):
    trial.suggest_categorical('regressor', ['Xgboost'])

    # 1. Heavily constrain tree complexity
    # We drop max depth significantly. XGBoost rarely needs to go past 8 or 10.
    max_depth = trial.suggest_int('xgb_max_depth', 3, 12)
    
    # We switch back to 'depthwise' to prevent 'lossguide' from making runaway asymmetric branches
    grow_policy = 'depthwise'
    
    # 2. Lower estimators, slightly higher learning rate for stability
    n_estimators = trial.suggest_int('xgb_n_estimators', 200, 1000, step=100)
    learning_rate = trial.suggest_float('xgb_learning_rate', 0.02, 0.2)

    # 3. Force Aggressive Subsampling (Feature & Row Dropping)
    # This forces trees to look at different columns and rows, preventing memorization
    subsample = trial.suggest_float('xgb_subsample', 0.60, 0.80)
    colsample_bytree = trial.suggest_float('xgb_colsample_bytree', 0.50, 0.75)
    colsample_bylevel = trial.suggest_float('xgb_colsample_bylevel', 0.50, 0.75)

    # 4. Drastically raise min_child_weight
    # Higher numbers mean a branch MUST contain a significant chunk of properties to exist
    min_child_weight = trial.suggest_int('xgb_min_child_weight', 15, 45)

    # 5. Crank up Regularization Penalties
    gamma = trial.suggest_float('xgb_gamma', 1.0, 9.0)
    reg_alpha = trial.suggest_float('xgb_reg_alpha', 0.01, 10.0, log=True)   # Stronger L1
    reg_lambda = trial.suggest_float('xgb_reg_lambda', 0.1, 20.0, log=True)  # Stronger L2

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        colsample_bylevel=colsample_bylevel,
        gamma=gamma,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        grow_policy=grow_policy,
        tree_method='hist',
        n_jobs=-1,
        random_state=42
    )

    # 6. Cross Validation Setup (Unchanged)
    cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_results = cross_validate(
        model,
        X_train, # Make sure this is your transformed data if you have categorical pipelines!
        y_train['Price(Cr)'],
        scoring='neg_mean_absolute_error',
        cv=cv_strategy,
        return_train_score=True,
        n_jobs=-1
    )

    # Convert scikit-learn's negative outputs into absolute positive numbers
    train_mae = abs(cv_results['train_score'].mean())
    val_mae = abs(cv_results['test_score'].mean())

    # 7. Log clear, positive attributes to Optuna Dashboard
    trial.set_user_attr('train_score_mean', train_mae)                     # Will show 0.102
    trial.set_user_attr('overfitting_gap', val_mae - train_mae)            # Will show 0.268

    # 8. Return positive metric to MINIMIZE
    return val_mae                                                         # Optuna will minimize 0.370 down towards 0

In [30]:
# Create your fresh study
study = optuna.create_study(direction='minimize', study_name="XGBoost_Final_Production")

# xgb_trial_154 = {
#     'regressor': 'Xgboost', 
#     'xgb_n_estimators': 400, 
#     'xgb_max_depth': 4, 
#     'xgb_learning_rate': 0.055641, 
#     'xgb_subsample': 0.792755, 
#     'xgb_colsample_bytree': 0.625803, 
#     'xgb_colsample_bylevel': 0.578621, 
#     'xgb_gamma': 3.183181, 
#     'xgb_min_child_weight': 23, 
#     'xgb_reg_alpha': 0.902825, 
#     'xgb_reg_lambda': 6.352308
# }
# study.enqueue_trial(xgb_trial_154)

# # Format Trial 4 perfectly for the encoder
# gold_star_trial = {
#     'regressor': 'Xgboost',
#     'xgb_n_estimators': 700,
#     'xgb_max_depth': 11,
#     'xgb_learning_rate': 0.023514,
#     'xgb_subsample': 0.525535,
#     'xgb_colsample_bytree': 0.675238,
#     'xgb_colsample_bylevel': 0.992857,
#     'xgb_gamma': 6.402284,
#     'xgb_min_child_weight': 6,
#     'xgb_reg_alpha': 0.063855,
#     'xgb_reg_lambda': 0.000568,
#     'xgb_grow_policy': 'lossguide',
#     'xgb_max_leaves': 53
# }

# Inject it into the study's memory
# study.enqueue_trial(gold_star_trial)


[I 2026-05-26 22:22:05,782] A new study created in memory with name: XGBoost_Final_Production


In [31]:
study.optimize(XGBoost_Anti_Overfit_Objective,n_trials=50)

[I 2026-05-26 22:22:08,321] Trial 0 finished with value: 1.5635244995313557 and parameters: {'regressor': 'Xgboost', 'xgb_max_depth': 9, 'xgb_n_estimators': 200, 'xgb_learning_rate': 0.12548235593640664, 'xgb_subsample': 0.711101021890531, 'xgb_colsample_bytree': 0.6884816907863609, 'xgb_colsample_bylevel': 0.7303548373816193, 'xgb_min_child_weight': 24, 'xgb_gamma': 4.360962316488263, 'xgb_reg_alpha': 0.041960974934824945, 'xgb_reg_lambda': 4.582241037732522}. Best is trial 0 with value: 1.5635244995313557.
[I 2026-05-26 22:22:09,280] Trial 1 finished with value: 1.6155479413452034 and parameters: {'regressor': 'Xgboost', 'xgb_max_depth': 12, 'xgb_n_estimators': 300, 'xgb_learning_rate': 0.1729975337939082, 'xgb_subsample': 0.6837342267724377, 'xgb_colsample_bytree': 0.5355518281594452, 'xgb_colsample_bylevel': 0.7374377004027222, 'xgb_min_child_weight': 23, 'xgb_gamma': 8.21941314510763, 'xgb_reg_alpha': 0.6097953198884892, 'xgb_reg_lambda': 11.171427940584278}. Best is trial 0 with 

In [32]:
df = study.trials_dataframe()
df.sort_values(by=['value','user_attrs_overfitting_gap'],ascending=[True,True]).head(3)

,number,value,datetime_start,datetime_complete,duration,params_regressor,params_xgb_colsample_bylevel,params_xgb_colsample_bytree,params_xgb_gamma,params_xgb_learning_rate,params_xgb_max_depth,params_xgb_min_child_weight,params_xgb_n_estimators,params_xgb_reg_alpha,params_xgb_reg_lambda,params_xgb_subsample,user_attrs_overfitting_gap,user_attrs_train_score_mean,state
46,46,1.504026,2026-05-26 22:23:26.371604,2026-05-26 22:23:27.480069,0 days 00:00:01.108465,Xgboost,0.686504,0.739567,4.325712,0.137341,10,16,300,0.186798,1.085081,0.792823,0.467595,1.036431,COMPLETE
21,21,1.504953,2026-05-26 22:22:43.377683,2026-05-26 22:22:45.063164,0 days 00:00:01.685481,Xgboost,0.659618,0.747688,4.520119,0.117633,10,15,600,0.080964,3.484151,0.772119,0.493700,1.011253,COMPLETE
13,13,1.507234,2026-05-26 22:22:29.016438,2026-05-26 22:22:30.600711,0 days 00:00:01.584273,Xgboost,0.656262,0.743013,4.270050,0.193918,9,15,600,0.078649,3.613851,0.798894,0.534513,0.972721,COMPLETE


In [33]:
study.best_params

{'regressor': 'Xgboost',
 'xgb_max_depth': 10,
 'xgb_n_estimators': 300,
 'xgb_learning_rate': 0.13734114985791407,
 'xgb_subsample': 0.7928228476451872,
 'xgb_colsample_bytree': 0.7395667303550625,
 'xgb_colsample_bylevel': 0.6865044423270638,
 'xgb_min_child_weight': 16,
 'xgb_gamma': 4.325712478129857,
 'xgb_reg_alpha': 0.18679827018441542,
 'xgb_reg_lambda': 1.0850805111427586}

In [34]:
print("Best value: ",study.best_value)

Best value:  1.5040262190839553


In [35]:
params = study.best_params
params.pop('regressor')

'Xgboost'

In [36]:
best_params = {}
for key,value in params.items():
    best_params[key[4:]] = value

In [37]:
best_params

{'max_depth': 10,
 'n_estimators': 300,
 'learning_rate': 0.13734114985791407,
 'subsample': 0.7928228476451872,
 'colsample_bytree': 0.7395667303550625,
 'colsample_bylevel': 0.6865044423270638,
 'min_child_weight': 16,
 'gamma': 4.325712478129857,
 'reg_alpha': 0.18679827018441542,
 'reg_lambda': 1.0850805111427586}

In [38]:
underfitted_best_params = {'max_depth': 8,
 'n_estimators': 700,
 'learning_rate': 0.03784249384284771,
 'subsample': 0.7836105782268438,
 'colsample_bytree': 0.7237677753352395,
 'colsample_bylevel': 0.5446839132083849,
 'min_child_weight': 15,
 'gamma': 1.359495501026701,
 'reg_alpha': 0.014393494799461814,
 'reg_lambda': 0.19305735637398955}

In [39]:
overfitted_params = {'n_estimators': 1000,
 'max_depth': 15,
 'learning_rate': 0.00505825958265549,
 'subsample': 0.8944998368111671,
 'colsample_bytree': 0.9305368372411382,
 'colsample_bylevel': 0.7407153424980754,
 'gamma': 0.021373237297058784,
 'min_child_weight': 4,
 'reg_alpha': 0.005116450915274579,
 'reg_lambda': 0.0007460273123214234,
 'grow_policy': 'lossguide',
 'max_leaves': 52}

In [52]:
best_model = XGBRegressor(**overfitted_params)
best_model.fit(X_train,y_train['Price(Cr)'])
y_pred = best_model.predict(X_test)
print('R2 Score ',r2_score(y_test,y_pred))
print('MAE ',mean_absolute_error(y_test,y_pred))

R2 Score  0.9351409077644348
MAE  1.3365603685379028


In [53]:
y_pred = best_model.predict(X_train)
print('R2 Score ',r2_score(y_train,y_pred))
print('MAE ',mean_absolute_error(y_train,y_pred))

R2 Score  0.9854008555412292
MAE  0.8770206570625305


In [54]:
import pickle

In [55]:
with open('model_houses.pkl','wb') as file:
    pickle.dump(best_model,file)

In [56]:
# Pipeline from Feature Selction/

In [57]:
from sklearn.preprocessing import OneHotEncoder,FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from category_encoders import TargetEncoder
from sklearn import set_config

set_config(transform_output='pandas')

In [58]:

preprocessor = ColumnTransformer(
    transformers=[
        
        ('TargetEncoding', TargetEncoder(smoothing=5, min_samples_leaf=2, handle_unknown='value'), ['Main Location', 'Building']),
        
        ('LogTransform', FunctionTransformer(np.log1p), ['Area(Marla)'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [59]:
X_train.columns

Index(['Main Location', 'Area(Marla)', 'Bath(s)', 'Bedroom(s)',
       'Servant Quarters', 'Kitchens', 'Store Rooms', 'Storey Unit',
       'IsPrimeLoc', 'SolarInstalled', 'WaterBore', 'CornerHouse',
       'Property era'],
      dtype='str')

In [60]:
import pickle

In [15]:
with open('model_flats.pkl','rb') as file:
    best_model = pickle.load(file)

In [16]:
best_model

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,0.7407153424980754
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9305368372411382
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.d

In [17]:
best_model.fit(X_train,y_train)
y_pred = best_model.predict(X_test)
print('R2 Score ',r2_score(y_test,y_pred))
print('MAE ',mean_absolute_error(y_test,y_pred))

R2 Score  0.9059523940086365
MAE  0.4135400950908661


In [18]:
y_pred = best_model.predict(X_train)
print('R2 Score ',r2_score(y_train,y_pred))
print('MAE ',mean_absolute_error(y_train,y_pred))

R2 Score  0.9921247363090515
MAE  0.17938539385795593


In [93]:
np.log1p(4)

np.float64(1.6094379124341003)

In [95]:
test_user = {     
    'Main Location':0.745359,     
    'Building': 0.796128,   
    'Area(Marla)': 1.6093147,
    'Property era': 3,
    'Floor Level': 2,
    'Building Type': 1,
    'Elevator Capacity': 1,
    'Bedroom(s)': 1, 
    'Bath(s)': 1, 
    'Kitchens': 1, 
    'Parking Spaces': 4, 
    'Servant Quarters': 1
}

In [94]:
X_train.loc[601]

Main Location        0.745359
Building             0.796128
Area(Marla)          1.609438
Bath(s)              1.000000
Bedroom(s)           1.000000
Servant Quarters     1.000000
Kitchens             2.000000
Parking Spaces       4.000000
Property era         3.000000
Floor Level          2.000000
Building Type        1.000000
Elevator Capacity    1.000000
Name: 601, dtype: float64

In [96]:
best_model.predict(pd.DataFrame(test_user,index=[0]))

array([0.37641874], dtype=float32)

In [100]:
test_user

Main Location        0.745359
Building             0.796128
Area(Marla)          1.609438
Bath(s)              1.000000
Bedroom(s)           1.000000
Servant Quarters     1.000000
Kitchens             2.000000
Parking Spaces       4.000000
Property era         3.000000
Floor Level          2.000000
Building Type        1.000000
Elevator Capacity    1.000000
Name: 601, dtype: float64

In [102]:
best_model.predict(test_user)

array([0.5286955 , 0.3867428 , 0.5628618 , 0.377966  , 0.41009933,
       0.39205548, 0.37182418, 0.377966  , 0.50186896, 0.6504266 ,
       0.38483036, 0.54090494, 0.59188896, 0.55631405, 0.7422054 ,
       0.424489  , 0.5142318 , 0.44177705, 1.0482271 , 0.37673244,
       1.833477  , 0.377966  , 0.41536918, 0.3815906 , 0.3898921 ,
       1.1334274 , 0.39342126, 0.56127244, 0.39486557, 0.38674188,
       0.42555723, 0.3815906 , 0.77097994, 0.41711193, 0.49631512,
       0.38064563, 0.73786277, 0.3818222 , 0.7041699 , 0.42987654,
       0.6274359 , 0.3842702 , 0.9162685 , 0.38829055, 0.42285398,
       0.6118032 , 0.5270103 , 0.38646072, 0.51012826, 0.4115586 ,
       0.4385189 , 0.44111964, 0.9485097 , 0.3815906 , 3.3446589 ,
       0.5189698 , 0.70114064, 0.3818222 , 0.5445043 , 0.5497705 ,
       0.3842702 , 0.42964682, 0.5519489 , 0.42308462, 0.99491626,
       0.538872  , 0.3842702 , 0.53877896, 0.5118972 , 0.38216445,
       0.7527551 , 0.7272596 , 0.51300776, 0.5118972 , 0.42496

In [103]:
y_train.loc[601:]

,Price(Cr)
601,1.60
2802,2.05
1687,3.30
2683,1.14
5901,0.98
...,...
4366,1.55
4931,1.95
3332,0.95
5650,0.36


In [101]:
test_user = X_train.loc[601:]

In [ ]:
property_era_mapping = {
    'Vintage': 1,
    'Established': 2,
    'Recently Built': 3,
    'Modern Era': 4,
    'Brand New (2025)': 5,
    'Future': 6
}

# 2. Elevation order: From lowest floors up to the top views
floor_level_mapping = {
    'Lower Floors': 1,
    'Ground/First Floor': 2,
    'Mid-Level': 3,
    'High-Rise': 4
}

# 3. Structural scale: From single-unit ground plots to mega-structures
building_type_mapping = {
    'Ground/Flat': 1,
    'Lower Levels': 2,
    'Mid Rise Core': 3,
    'High Rise': 4,
    'Skyscarper': 5
}

# 4. Premium amenity order: From basic access to heavy-duty infrastructures
elevator_capacity_mapping = {
    'Single/None': 1,
    'Dual Setup': 2,
    'Multiple Bank': 3,
    'High-Capacity': 4
}


In [83]:
X_train_pipeline = pd.read_csv('../Missing Values Imputation/Flats/X_train.csv',index_col=0).drop(columns=['Price per Unit','Elevators','Floor in Building','Floor','Store Rooms','luxury_type'])
X_test_pipeline = pd.read_csv('../Missing Values Imputation/Flats/X_test.csv',index_col=0).drop(columns=['Price per Unit','Elevators','Floor in Building','Floor','Store Rooms','luxury_type'])
y_train_pipeline = pd.read_csv('../Missing Values Imputation/Flats/y_train.csv',index_col=0)
y_test_pipeline = pd.read_csv('../Missing Values Imputation/Flats/y_test.csv',index_col=0)

In [84]:
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor

# ==========================================
# 1. DEFINE CUSTOM TRANSFORMERS
# ==========================================

# Transformer for your custom ordinal dictionary scales
class CustomOrdinalMapper(BaseEstimator, TransformerMixin):
    def __init__(self, mappings):
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        for col, mapping_dict in self.mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(mapping_dict).fillna(1).astype(int)
        return X_copy

# Transformer for your log transformations that preserves DataFrames
class DataFrameLogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        for col in self.columns:
            if col in X_copy.columns:
                X_copy[col] = np.log1p(X_copy[col].astype(float))
        return X_copy


# ==========================================
# 2. SET UP CONFIGURATIONS & MASTER PIPELINE
# ==========================================

all_mappings = {
    'Property era': {
        'Vintage': 1, 'Established': 2, 'Recently Built': 3, 
        'Modern Era': 4, 'Brand New (2025)': 5, 'Future': 6
    },
    'Floor Level': {
        'Lower Floors': 1, 'Ground/First Floor': 2, 'Mid-Level': 3, 'High-Rise': 4
    },
    'Building Type': {
        'Ground/Flat': 1, 'Lower Levels': 2, 'Mid Rise Core': 3, 
        'High Rise': 4, 'Skyscarper': 5
    },
    'Elevator Capacity': {
        'Single/None': 1, 'Dual Setup': 2, 'Multiple Bank': 3, 'High-Capacity': 4
    }
}

# Assemble a clean, linear, pure DataFrame pipeline
production_pipeline = Pipeline([
    # Step 1: Map manual categorical text hierarchies to ordered integers
    ('ordinal_maps', CustomOrdinalMapper(all_mappings)),
    
    # Step 2: Target Encode location data (EXPLICITLY declaring cols here fixes the bug)
    ('target_encode', TargetEncoder(
        cols=['Main Location', 'Building'], 
        smoothing=20, 
        min_samples_leaf=2, 
        handle_unknown='value'
    )),
    
    # Step 3: Log transform the area column smoothly inline
    ('log_transform', DataFrameLogTransformer(columns=['Area(Marla)'])),
    
    # Step 4: Your elite production XGBoost model state
    ('model', best_model)
])


# ==========================================
# 3. TRAIN AND PICKLE
# ==========================================

# Fit the entire pipeline on your completely RAW, untouched training dataframe
print("Training production pipeline...")
production_pipeline.fit(X_train_pipeline, y_train_pipeline['Price(Cr)'])

# Save the master pipeline to disk
joblib.dump(production_pipeline, 'pak_real_estate_pipeline.pkl')
print("Success! Saved as pak_real_estate_pipeline.pkl")

Training production pipeline...
Success! Saved as pak_real_estate_pipeline.pkl


In [81]:
X_train

,Main Location,Building,Area(Marla),Bath(s),Bedroom(s),Servant Quarters,Kitchens,Parking Spaces,Property era,Floor Level,Building Type,Elevator Capacity
6148,0.835680,0.835680,0.693147,1.0,1.0,1.0,1.0,1.0,3,2,3,1
4031,0.682126,0.688180,1.098612,1.0,1.0,0.0,1.0,1.0,4,2,2,1
1222,0.788549,0.800230,1.609438,3.0,3.0,1.0,1.0,1.0,3,2,1,1
1783,0.756197,0.800029,1.945910,2.0,2.0,1.0,1.0,1.0,3,2,1,3
1550,0.746192,0.746192,1.386294,2.0,2.0,1.0,1.0,1.0,3,2,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4366,1.020540,1.033151,1.386294,1.0,1.0,1.0,1.0,4.0,5,3,3,2
4931,1.113525,1.113525,1.945910,2.0,3.0,1.0,1.0,1.0,3,2,2,1
3332,0.986039,0.837520,1.098612,1.0,1.0,0.0,1.0,1.0,3,1,3,2
5650,0.763765,0.763765,1.098612,1.0,1.0,1.0,2.0,4.0,3,1,3,2


In [ ]:
production_pipeline.predict(test)

In [90]:
# Load the newly exported asset
real_estate_ai = joblib.load('pak_real_estate_pipeline.pkl')

def get_valuation(user_form_data):
    # Convert input dict to a 1-row DataFrame
    df_input = pd.DataFrame([user_form_data])
    
    # Predict directly using the streamlined pipeline
    prediction = real_estate_ai.predict(df_input)
    return round(prediction[0], 2)

# Your exact test user profile
test_user = {     
    'Main Location': 'B-17',     
    'Building': 'B-17',   
    'Area(Marla)': 4,
    'Property era': 'Recently Built',
    'Floor Level': 'Ground/First Floor',
    'Building Type': 'Ground/Flat',
    'Elevator Capacity': 'Single/None',
    'Bedroom(s)': 1, 
    'Bath(s)': 1, 
    'Kitchens': 2, 
    'Parking Spaces': 4, 
    'Servant Quarters': 1
}

print(f"Predicted Market Price: {get_valuation(test_user)} Crores")

Predicted Market Price: 0.5899999737739563 Crores


In [32]:
pd.DataFrame(test_user,index=[0])

,Main Location,Building,Area(Marla),Property era,Floor Level,Building Type,Elevator Capacity,Bedroom(s),Bath(s),Kitchens,Parking Spaces,Servant Quarters
0,F-8,The Centaurus,1000.0,Brand New (2025),Ground/First Floor,Ground/Flat,Single/None,30,20,6,5,2


In [48]:

X_for_prediction = pd.read_csv('../Missing Values Imputation/Flats/X_train.csv',index_col=0).drop(columns=['Price per Unit','Elevators','Floor in Building','Floor','Store Rooms','luxury_type'])

In [70]:
X_for_prediction.sample(3)

,Bath(s),Area(Marla),Bedroom(s),Servant Quarters,Kitchens,Building,Main Location,Parking Spaces,Property era,Floor Level,Building Type,Elevator Capacity
601,1.0,4.0,1.0,1.0,2.0,B-17,B-17,4.0,Recently Built,Ground/First Floor,Ground/Flat,Single/None
3142,2.0,3.0,2.0,1.0,1.0,I-12,I-12,1.0,Brand New (2025),Lower Floors,Mid Rise Core,Dual Setup
5064,2.0,6.0,2.0,0.0,1.0,One Constitution Avenue,Constitution Avenue,1.0,Recently Built,Ground/First Floor,Ground/Flat,Single/None


In [92]:
y_train.loc[601]

Price(Cr)    1.6
Name: 601, dtype: float64

In [73]:
test_user

Bath(s)                                  2.0
Area(Marla)                              6.0
Bedroom(s)                               2.0
Servant Quarters                         0.0
Kitchens                                 1.0
Building             One Constitution Avenue
Main Location            Constitution Avenue
Parking Spaces                           1.0
Property era                  Recently Built
Floor Level               Ground/First Floor
Building Type                    Ground/Flat
Elevator Capacity                Single/None
Name: 5064, dtype: object

In [74]:
get_valuation(test_user)

np.float32(3.97)

In [79]:
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor

# ==========================================
# 1. DEFINE CUSTOM TRANSFORMERS
# ==========================================

# Transformer for your custom ordinal dictionary scales
class CustomOrdinalMapper(BaseEstimator, TransformerMixin):
    def __init__(self, mappings):
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        for col, mapping_dict in self.mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(mapping_dict).fillna(1).astype(int)
        return X_copy

# Transformer for your log transformations that preserves DataFrames
class DataFrameLogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        for col in self.columns:
            if col in X_copy.columns:
                X_copy[col] = np.log1p(X_copy[col].astype(float))
        return X_copy


# ==========================================
# 2. SET UP CONFIGURATIONS & MASTER PIPELINE
# ==========================================

all_mappings = {
    'Property era': {
        'Vintage': 1, 'Established': 2, 'Recently Built': 3, 
        'Modern Era': 4, 'Brand New (2025)': 5, 'Future': 6
    },
    'Floor Level': {
        'Lower Floors': 1, 'Ground/First Floor': 2, 'Mid-Level': 3, 'High-Rise': 4
    },
    'Building Type': {
        'Ground/Flat': 1, 'Lower Levels': 2, 'Mid Rise Core': 3, 
        'High Rise': 4, 'Skyscarper': 5
    },
    'Elevator Capacity': {
        'Single/None': 1, 'Dual Setup': 2, 'Multiple Bank': 3, 'High-Capacity': 4
    }
}

# Assemble a clean, linear, pure DataFrame pipeline
production_pipeline = ColumnTransformer([
    # Step 1: Map manual categorical text hierarchies to ordered integers
    ('ordinal_maps', CustomOrdinalMapper(all_mappings)),
    
    # Step 2: Target Encode location data (EXPLICITLY declaring cols here fixes the bug)
    ('target_encode', TargetEncoder(
        cols=['Main Location', 'Building'], 
        smoothing=20, 
        min_samples_leaf=2, 
        handle_unknown='value'
    )),
    
    # Step 3: Log transform the area column smoothly inline
    ('log_transform', DataFrameLogTransformer(columns=['Area(Marla)']))
])

In [80]:
production_pipeline.transform(X_for_prediction,)

NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.